# Chapitre 3 · Comment une machine « apprend »

Notebook du chapitre 3 de *Construire un LLM de zéro*. Tu démontes ici le mystère
du `loss.backward()` du chapitre 1 : un **capteur de pente** codé en trois lignes
de Python, la descente qui fait fondre une erreur, le gradient à deux boutons, et
la règle de la chaîne vérifiée au capteur.

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et
prêt à exécuter. Lis, exécute, modifie pour voir. À la fin, la section
**Exercices** : quatre défis à trous, du plus simple au plus costaud, validés
par des `assert`.

Tout tourne **sans GPU, sans connexion internet et sans aucune bibliothèque** :
du Python pur, rien d'autre.

## 1. Le mystère laissé au chapitre 1

La loss de MiniLM partait de 4.42 et finissait à 0.75, sans que personne ne touche
aux 91 497 nombres du modèle. Deux lignes ont tout fait :

```python
loss.backward()      # comment sait-il quoi corriger ?
optimiseur.step()    # et de combien ?
```

Ce chapitre perce le mystère. Et comme toujours, on dégonfle le problème avant de
l'attaquer : 91 497 nombres, c'est trop ; un seul, c'est parfait.

### 1.1 · Un seul nombre à régler

Sètondji, conducteur de zémidjan à Cotonou, veut un tarif simple :
`prix = w * distance`, avec un seul nombre `w` à choisir (des francs CFA par km).
Pour le régler, il dispose de quatre trajets récents dont il connaît la distance
et le prix que le client a accepté de payer.

La fonction `erreur(w)` mesure à quel point un candidat `w` se trompe **en
moyenne** sur ces quatre trajets : pour chaque trajet, l'écart entre le prix
prédit et le prix réel, au carré (pour que les écarts négatifs comptent aussi),
puis la moyenne des quatre. C'est la même idée que la loss du chapitre 1 :
un chiffre unique qui dit « tu te trompes de tant ».

In [ ]:
distances = [2.0, 5.0, 8.0, 12.0]            # km parcourus
prix      = [400.0, 700.0, 1250.0, 1650.0]   # FCFA payes par le client


def erreur(w):
    """Erreur moyenne du modele  prix_predit = w * distance  sur les 4 trajets."""
    total = 0.0
    for d, p in zip(distances, prix):
        ecart = w * d - p        # prix predit moins prix reel
        total += ecart ** 2      # au carre : les ecarts negatifs comptent aussi
    return total / len(distances)


# le .replace : la virgule de Python devient une espace, séparateur de milliers à la française
print(f"erreur(100) = {erreur(100.0):>9,.0f}".replace(",", " "))
print(f"erreur(200) = {erreur(200.0):>9,.0f}".replace(",", " "))

Ces valeurs sont en « FCFA au carré », une unité qui ne parle à personne. Peu
importe : elles ne servent qu'à **comparer**. 121 250 < 193 750, donc `w = 100`
se trompe moins que `w = 200`. C'est tout ce qu'on demande à une erreur.

### 1.3 · Première stratégie : tâtonner

La stratégie la plus honnête du monde : essayer des valeurs et garder la meilleure.

In [ ]:
# Premiere strategie : tatonner. On essaie des valeurs et on garde la meilleure.
for w in [100.0, 120.0, 140.0, 160.0, 180.0, 200.0]:
    print(f"w = {w:5.0f} FCFA/km  ->  erreur = {erreur(w):>9,.0f}".replace(",", " "))

L'erreur descend jusqu'à `w = 140`, puis remonte : le bon tarif se cache autour
de 140 FCFA/km. Avec un bouton, tâtonner marche. Avec les 91 497 boutons de
MiniLM, le tâtonnage explose : il nous faut un capteur qui, sans rien essayer
d'autre, dit dans quel sens pousser `w` depuis là où on est.

## 2. Le capteur de pente

L'idée tient en une phrase : **pousse l'entrée d'un tout petit `h`, et regarde de
combien la sortie bouge**. Le rapport `(sortie gagnée) / (entrée poussée)` est la
pente locale. Essaie-la à la main sur `f(x) = x ** 2` au point `x = 3` :

In [ ]:
# Le geste, a la main : pousse l'entree d'un cheveu, regarde la sortie.
f = lambda x: x ** 2
h = 0.001
pente_en_3 = (f(3 + h) - f(3)) / h        # (sortie gagnee) / (entree poussee)
print(f"f(3) = {f(3)}   f(3 + h) = {f(3 + h)}")
print(f"pente en 3 : {pente_en_3:.3f}")   # la sortie bouge environ 6 fois plus vite

Une amélioration avant d'en faire une fonction : plutôt que de pousser d'un seul
côté, on mesure **des deux côtés** du point, en `x + h` et `x - h`. La mesure est
centrée sur le point, et nettement plus précise. La largeur totale du pas devient
`2 * h`, d'où le dénominateur :

In [ ]:
def pente(f, x, h=1e-5):
    """De combien f(x) bouge-t-elle quand on pousse x d'un cheveu ?"""
    return (f(x + h) - f(x - h)) / (2 * h)

In [ ]:
p = pente(lambda x: x ** 2, 3.0)    # lambda : la fonction carre, en une expression
print(p)                            # 6.0 : exactement le calcul manuel. Il fonctionne.

### 2.2 · Le capteur, branché sur la courbe de Sètondji

Le **signe** donne la direction (pente négative : quand `w` monte, l'erreur
descend), la **taille** donne l'urgence (raide : on est loin du compte ; presque
plat : le fond est tout près). Règle unique : pousser `w` à l'opposé du signe.

In [ ]:
# Lisons le capteur sur la courbe d'erreur de Setondji.
for w in [100.0, 200.0, 144.0]:
    print(f"w = {w:5.0f}  ->  pente = {pente(erreur, w):>8.1f}")

Ce que tu viens de coder, les mathématiciens l'appellent la **dérivée**. Et il
existe une seconde façon d'obtenir une pente : la calculer **exactement** par une
formule (l'arsenal des dérivées analytiques : constante → 0, $x$ → 1,
$x^2$ → $2x$, $x^n$ → $n\,x^{n-1}$). Le réflexe de métier : l'exacte pour la
vitesse, le capteur pour la vérification (le *gradient checking*).

### 2.4 · Le capteur a ses limites : le choix de h

Trop grand, la mesure est biaisée (on mesure la pente d'une corde, pas celle du
point). Ridiculement petit, les arrondis de l'ordinateur mangent la mesure. La
zone `1e-4` à `1e-6` est un bon compromis. La preuve sur `f(x) = x**3`, dont la
pente exacte en `x = 3` vaut 27 :

In [ ]:
# Cas qui echoue : un h trop grand ou trop petit fausse le capteur.
f = lambda x: x ** 3                 # pente exacte en x = 3 : 27
for h in [1.0, 1e-5, 1e-13]:
    estimation = (f(3 + h) - f(3 - h)) / (2 * h)
    print(f"h = {h:g}  ->  pente estimee : {estimation}")

## 3. Suivre la pente : la descente

Le capteur en main, la stratégie s'écrit en une ligne, répétée en boucle :
`w = w - lr * pente(f, w)`. Le signe moins fait descendre (on va à l'opposé de
la montée), et `lr` (le learning rate, le taux d'apprentissage) règle la taille
du pas.

In [ ]:
def descendre(f, w, lr=0.005, n_pas=20):
    """Repete des pas de descente et renvoie le w final."""
    for etape in range(n_pas):
        if etape % 4 == 0:
            print(f"etape {etape:2d} | w = {w:7.2f} | erreur = {f(w):>10,.0f}".replace(",", " "))
        w = w - lr * pente(f, w)
    print(f"final    | w = {w:7.2f} | erreur = {f(w):>10,.0f}".replace(",", " "))
    return w


w_final = descendre(erreur, w=50.0)

L'erreur part de 529 375 et fond jusqu'à 7 157, le fond de la vallée : Sètondji
tient son tarif, environ 144 FCFA le kilomètre. Et remarque une élégance que
personne n'a programmée : les pas rétrécissent tout seuls, car le pas vaut
`lr * pente` et la pente s'aplatit près du fond.

### 3.3 · Le cas qui échoue : le pas trop grand

Le learning rate est un réglage délicat. Trop grand, chaque pas saute par-dessus
le fond de la vallée et atterrit plus haut que le point de départ : l'erreur
**explose** au lieu de fondre. Regarde (aucune exception n'est levée, le code
tourne : c'est justement le piège, la divergence s'affiche mais ne plante pas) :

In [ ]:
# Cas qui echoue : learning rate trop grand, la descente diverge.
w = 50.0
for etape in range(8):
    print(f"etape {etape} | w = {w:8.1f} | erreur = {erreur(w):>13,.0f}".replace(",", " "))
    w = w - 0.02 * pente(erreur, w)     # lr = 0.02 : 4 fois trop grand
print("L'erreur EXPLOSE : chaque pas saute par-dessus la vallee, de plus en plus haut.")

## 4. Plusieurs boutons à la fois : le gradient

Le plancher de 7 157 n'est pas un hasard : la distance seule n'explique pas tout,
les embouteillages comptent aussi. On enrichit le modèle avec la durée du trajet :
`prix = w1 * distance + w2 * duree`. Deux boutons à régler, donc deux pentes à
mesurer : on pousse `w1` en gelant `w2`, puis l'inverse (les **dérivées
partielles**). La liste des deux pentes, c'est le **gradient**.

In [ ]:
durees = [15.0, 12.0, 30.0, 28.0]        # minutes (les embouteillages varient !)


def erreur2(w1, w2):
    """Erreur moyenne du modele  prix_predit = w1 * distance + w2 * duree."""
    total = 0.0
    for d, t, p in zip(distances, durees, prix):
        ecart = w1 * d + w2 * t - p
        total += ecart ** 2
    return total / len(distances)


# Notre meilleur w d'avant, et la duree ignoree (w2 = 0) : meme erreur qu'avant.
print(f"erreur2(143.88, 0) = {erreur2(143.88, 0.0):,.0f}".replace(",", " "))

In [ ]:
def gradient_numerique(f, w1, w2, h=1e-5):
    """Les deux pentes de f au point (w1, w2) : une par bouton."""
    p1 = (f(w1 + h, w2) - f(w1 - h, w2)) / (2 * h)   # on pousse w1, w2 est gele
    p2 = (f(w1, w2 + h) - f(w1, w2 - h)) / (2 * h)   # on pousse w2, w1 est gele
    return p1, p2

In [ ]:
# Validons le capteur double sur une fonction dont l'arsenal donne les pentes
# exactes : sur g(w1, w2) = w1**2 + 3*w1*w2, au point (2, 1), les pentes valent
# 2*w1 + 3*w2 = 7 (selon w1) et 3*w1 = 6 (selon w2).
g = lambda w1, w2: w1 ** 2 + 3 * w1 * w2
p1, p2 = gradient_numerique(g, 2.0, 1.0)
print(f"({p1:.4f}, {p2:.4f})")    # (7.0000, 6.0000) : le capteur double est juste

In [ ]:
# Et sur l'erreur de Setondji, au point exact ou la section 3 nous a laisses :
p1, p2 = gradient_numerique(erreur2, 143.88, 0.0)
print(f"gradient de erreur2 en (143.88, 0) : ({p1:.1f}, {p2:.1f})")
print("Le bouton w1 est presque a plat ; c'est le bouton w2 (la duree) qui compte.")

### 4.4 · La descente à deux boutons, et au-delà

La règle de mise à jour ne change pas d'un iota : chaque bouton fait son pas,
avec sa pente à lui.

In [ ]:
# La descente, identique, mais sur les DEUX boutons a la fois.
w1, w2, lr = 143.88, 0.0, 0.001
for etape in range(301):
    p1, p2 = gradient_numerique(erreur2, w1, w2)
    if etape % 50 == 0:
        print(f"etape {etape:3d} | w1 = {w1:6.2f} | w2 = {w2:5.2f} | erreur = {erreur2(w1, w2):>7,.0f}".replace(",", " "))
    w1 = w1 - lr * p1
    w2 = w2 - lr * p2
print(f"final     | w1 = {w1:6.2f} | w2 = {w2:5.2f} | erreur = {erreur2(w1, w2):>7,.0f}".replace(",", " "))

Le plancher de 7 157 est pulvérisé : l'erreur tombe à 50, et la descente a
réparti les rôles toute seule (environ 111 FCFA/km + 12 FCFA/min). Un problème,
pourtant : le capteur numérique coûte deux évaluations **par bouton**. Pour
MiniLM, près de 183 000 passages du modèle pour une seule étape. Il nous faut
toutes les pentes, exactes, pour le prix d'un seul passage : la règle de la
chaîne.

## 5. La règle de la chaîne

Notre erreur est une **chaîne** de transformations : `w` devient un prix prédit
(`w * distance`), le prix devient un écart (`prix_predit - prix_reel`), l'écart
devient une erreur (`ecart ** 2`). La règle de la chaîne affirme que la pente de
la chaîne entière est le **produit des pentes de chaque maillon**. Vérifions-le
au capteur, sur un seul trajet (8 km, 1 250 FCFA), au point `w0 = 100`.

In [ ]:
d, p_reel = 8.0, 1250.0                  # le troisieme trajet de Setondji
carre = lambda e: e ** 2                 # le dernier maillon de la chaine


def prix_pred(w):
    return w * d                          # maillon 1 : w -> prix predit


def erreur_trajet(w):
    return carre(prix_pred(w) - p_reel)   # la chaine entiere : w -> erreur


w0 = 100.0
ecart0 = prix_pred(w0) - p_reel           # l'ecart au point ou l'on se trouve
globale = pente(erreur_trajet, w0)        # la pente de la chaine entiere
print(f"ecart au point w0 = {ecart0}")
print(f"pente globale (mesuree au capteur) = {globale:.1f}")

In [ ]:
# On mesure la pente de CHAQUE maillon avec le capteur, puis on multiplie.
locale_1 = pente(prix_pred, w0)          # maillon 1 : w -> prix predit (pente = 8, la distance)
locale_2 = pente(carre, ecart0)          # maillon 2 : ecart -> ecart**2 (pente = 2 * ecart)
produit = locale_1 * locale_2
print(f"pente du maillon 1 : {locale_1:.4f}")
print(f"pente du maillon 2 : {locale_2:.4f}")
print(f"produit des deux   : {produit:.1f}")

Le produit, $8 \times (-900) = -7200$, retombe exactement sur la pente globale :
la règle de la chaîne tient. Note que le maillon « au carré » se mesure **en
`ecart0`**, pas en `w0` : chaque maillon se dérive au point où lui se trouve
quand la donnée le traverse. Parcourue de la sortie vers l'entrée, cette
multiplication s'appelle la **backpropagation** : le `backward` de
`loss.backward()`, littéralement.

## Exercices

À toi de jouer : quatre exercices, du plus simple (●) au plus costaud (●●●).
Chaque exercice te fait réécrire de mémoire une pièce maîtresse de la leçon.
Chaque cellule marquée `# TODO(toi)` contient un trou ; complète-le, puis exécute
la cellule de validation (`assert`) qui suit : si elle passe sans erreur, c'est
gagné. Exécute d'abord la leçon en entier, les exercices s'appuient sur ses
données.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta
place. Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi.
Les réponses sont dans le notebook solution, à n'ouvrir qu'après avoir vraiment
essayé.

### Exercice 1 · Le capteur de pente — niveau ●

Trois lignes qui portent tout le chapitre. Réécris le capteur : pousse `x` des
deux côtés du point (`x + h` et `x - h`), et fais le rapport sur la largeur
totale du pas.

In [ ]:
def pente(f, x, h=1e-5):
    """De combien f(x) bouge-t-elle quand on pousse x d'un cheveu ?"""
    # TODO(toi) : mesure la pente des deux cotes du point.
    # 1) evalue f en x + h, puis en x - h ;
    # 2) renvoie leur difference divisee par 2*h (la largeur totale du pas).
    return ...

In [ ]:
# Validation : le capteur de pente.
p = pente(lambda x: x ** 2, 3.0)
assert isinstance(p, float), "le capteur doit renvoyer un nombre"
assert abs(p - 6.0) < 1e-4, f"pente attendue 6.0 en x=3 pour x**2, obtenue {p}"
assert pente(erreur, 100.0) < 0, "en w=100 la pente doit etre negative (il faut monter)"
assert pente(erreur, 200.0) > 0, "en w=200 la pente doit etre positive (il faut descendre)"
print("Capteur OK : pente(x**2, en 3) =", round(p, 6))

### Exercice 2 · Un pas de descente — niveau ●●

La descente répète un petit pas à l'opposé de la pente. Complète le pas manquant
dans `descendre` : mesure la pente de `f` en `w` avec le capteur, puis avance à
l'opposé, avec un pas de taille `lr`.

In [ ]:
def descendre(f, w, lr=0.005, n_pas=20):
    """Repete des pas de descente et renvoie le w final."""
    for etape in range(n_pas):
        if etape % 4 == 0:
            print(f"etape {etape:2d} | w = {w:7.2f} | erreur = {f(w):>10,.0f}".replace(",", " "))
        # TODO(toi) : un pas de descente.
        # 1) mesure la pente de f en w avec le capteur : pente(f, w) ;
        # 2) avance a l'oppose de la pente : le nouveau w vaut  w - lr * pente.
        w = ...
    print(f"final    | w = {w:7.2f} | erreur = {f(w):>10,.0f}".replace(",", " "))
    return w


w_final = descendre(erreur, w=50.0)

In [ ]:
# Validation : la descente doit converger vers w* (environ 143.9 FCFA/km).
assert isinstance(w_final, float), "descendre doit renvoyer le w final"
assert abs(w_final - 143.9) < 0.5, f"w final attendu proche de 143.9, obtenu {w_final:.2f}"
assert erreur(w_final) < 7200, "l'erreur finale doit etre au plancher (environ 7 157)"
print(f"Descente OK : w = {w_final:.2f} FCFA/km, erreur = {erreur(w_final):,.0f}".replace(",", " "))

### Exercice 3 · Le gradient numérique — niveau ●●

Deux boutons, donc deux capteurs : pousse `w1` des deux côtés en gelant `w2`,
puis l'inverse. Renvoie les deux pentes.

In [ ]:
def gradient_numerique(f, w1, w2, h=1e-5):
    """Les deux pentes de f au point (w1, w2) : une par bouton."""
    # TODO(toi) : deux capteurs, un par bouton.
    # pente selon w1 : pousse w1 des deux cotes, GELE w2
    #   -> (f(w1 + h, w2) - f(w1 - h, w2)) / (2 * h)
    # pente selon w2 : gele w1, pousse w2 des deux cotes.
    p1 = ...
    p2 = ...
    return p1, p2

In [ ]:
# Validation : sur g(w1, w2) = w1**2 + 3*w1*w2, les pentes exactes
# au point (2, 1) valent 2*w1 + 3*w2 = 7 (selon w1) et 3*w1 = 6 (selon w2).
g = lambda w1, w2: w1 ** 2 + 3 * w1 * w2
p1, p2 = gradient_numerique(g, 2.0, 1.0)
assert abs(p1 - 7.0) < 1e-4, f"pente selon w1 attendue 7.0, obtenue {p1}"
assert abs(p2 - 6.0) < 1e-4, f"pente selon w2 attendue 6.0, obtenue {p2}"
print(f"Gradient OK : ({p1:.4f}, {p2:.4f})")

# Et sur l'erreur de Setondji, au point (143.88, 0) :
p1, p2 = gradient_numerique(erreur2, 143.88, 0.0)
print(f"gradient de erreur2 en (143.88, 0) : ({p1:.1f}, {p2:.1f})")
print("Le bouton w1 est presque a plat ; c'est le bouton w2 (la duree) qui compte.")

### Exercice 4 · La règle de la chaîne — niveau ●●●

La pente de la chaîne entière doit être le produit des pentes des maillons.
Mesure au capteur la pente de chaque maillon de la section 5 (`prix_pred` au
point `w0`, `carre` au point `ecart0`), puis multiplie et compare à la pente
globale déjà mesurée.

In [ ]:
# TODO(toi) : mesure la pente de CHAQUE maillon avec le capteur, puis multiplie.
# maillon 1 : w -> prix predit.   pente locale = pente(prix_pred, w0)
# maillon 2 : ecart -> ecart**2.  pente locale = pente(carre, ecart0)
locale_1 = ...
locale_2 = ...
produit = ...
print(f"pente du maillon 1 : {locale_1:.4f}")
print(f"pente du maillon 2 : {locale_2:.4f}")
print(f"produit des deux   : {produit:.1f}")

In [ ]:
# Validation : produit des pentes locales = pente globale.
assert abs(locale_1 - 8.0) < 1e-3, f"maillon 1 : pente attendue 8.0, obtenue {locale_1}"
assert abs(locale_2 - 2 * ecart0) < 1e-2, f"maillon 2 : pente attendue {2 * ecart0}, obtenue {locale_2}"
assert abs(produit - globale) < 0.1, (
    f"le produit des pentes locales ({produit:.2f}) doit egaler la pente globale ({globale:.2f})"
)
print("Regle de la chaine OK : le produit des pentes locales retombe sur la pente globale.")

## Et maintenant ?

Tu possèdes les trois idées du chapitre 4 : le capteur de pente (la dérivée), la
liste des pentes (le gradient), et la multiplication le long de la chaîne (la
règle de la chaîne). Au chapitre 4, tu construis la machine qui applique tout ça
**automatiquement**, à n'importe quelle chaîne de calculs : ton propre moteur
d'autograd, celui qui se cache derrière le `loss.backward()` du chapitre 1.
C'est le premier grand rendez-vous **IA débranchée** du livre : personne d'autre
que toi n'écrira ce code.